In [ ]:
import torch
from torch import nn
import pandas as pd
from transformers import BartTokenizer, BartModel
from tqdm import tqdm

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
bart_model = BartModel.from_pretrained("facebook/bart-base")


# Defining the embedding function to convert sentences to numeric representations
def convert_to_embeddings(messages):
    embeddings_list = []
    for message in tqdm(messages):
        out = tokenizer(
            [message],
            padding=True,
            max_length=512,
            truncation=True,
            return_tensors="pt",
        )

        with torch.no_grad():
            bart_model.eval()

            pred = bart_model(
                input_ids=out["input_ids"], attention_mask=out["attention_mask"]
            )
            embeddings = pred.last_hidden_state.mean(dim=1).reshape((-1))
            embeddings_list.append(embeddings)
    return torch.stack(embeddings_list)

In [ ]:
# This dataset is licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) license.
# This allows for the sharing and adaptation of the datasets for any purpose, provided that the appropriate credit is given.
# Data downloaded from:
# https://archive.ics.uci.edu/dataset/228/sms+spam+collection


df = pd.read_csv("path_to_file", sep="\t", names=["type", "message"])

df["spam"] = df["type"] == "spam"
df.drop("type", axis=1, inplace=True)

In [ ]:
# seperating the data into training and validation datasets

df_train = df.sample(frac=0.8, random_state=0)
df_val = df.drop(index=df_train.index)

In [ ]:
X_train = convert_to_embeddings(df_train["message"].tolist())
X_val = convert_to_embeddings(df_val["message"].tolist())

In [ ]:
Y_train = torch.tensor(df_train["spam"].values, dtype=torch.float32).reshape((-1, 1))

Y_val = torch.tensor(df_val["spam"].values, dtype=torch.float32).reshape((-1, 1))

In [ ]:
# defining the model and training process using a single binary classifier neuron

model = nn.Linear(768, 1)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for i in range(0, 20000):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = loss_fn(outputs, Y_train)
    loss.backward()
    optimizer.step()

    if i % 1000 == 0:
        print(loss)

In [ ]:
# Evaluating the model on validation dataset and inspecting the metrics
def evaluate_model(X, Y):
    model.eval()
    with torch.no_grad():
        Y_pred = (
            nn.functional.sigmoid(model(X)) > 0.50
        )  # this value could be anything from experiments
        print("accuracy:", (Y_pred == Y).type(torch.float32).mean())

        print(
            "precision:",
            (Y_pred[Y_pred == 1] == Y[Y_pred == 1]).type(torch.float32).mean(),
        )

        print("sensitivity:", (Y_pred[Y == 1] == Y[Y == 1]).type(torch.float32).mean())

        print("specificity:", (Y_pred[Y == 0] == Y[Y == 0]).type(torch.float32).mean())


print("Evaluating on the training data")
evaluate_model(X_train, Y_train)

print("Evaluating on the validation data")
evaluate_model(X_val, Y_val)

In [ ]:
# Further evaluating on custon test dataset

X_custom = convert_to_embeddings(
    [
        "Please forward this message to the boss.",
        "You are the lucky Winner! Call us to get your prize",
        "Please contact me as soon as possible!",
    ]
)

model.eval()
with torch.no_grad():
    pred = nn.functional.sigmoid(model(X_custom))
    print(pred)